# 4. The 43 core motifs, start to end (Fig. 2a-c)

Each of the 116 biosamples has a vocabulary of de novo promoter motifs (notebook 02). This notebook derives the **43 core motifs** that recur across biosamples, running the paper's pipeline (`code/motif_pipeline/`) from its inputs to its output:

| step | script | what | time | run here |
|---|---|---|---|---|
| 1 | `01_ic_filter.py` | per biosample, from the 2,560 promoter PWMs of each of its ten models: keep the 750 highest-weight filters with information content > 10 bits -> ≤ 7,500 PWMs | 1 min for all 116 | yes |
| 3 | `03_cell_pipeline.py BSSID` | per biosample: TOMTOM of the set against itself and against JASPAR; connect similar PWMs (E ≤ 0.01, p ≤ 0.01) and cluster the similarity graph; keep clusters matching a JASPAR motif whose TF is expressed; merge clusters sharing a TF; one representative PWM per cluster -> median 73 motifs | TOMTOM hours + clustering 1-3 h **per biosample** | optional (K562; shipped TOMTOM tables reused) |
| 4 | `04_collect_features.py` | pool the representatives of the 116 biosamples -> 8,441 PWMs | seconds | yes |
| 5 | `05_self_tomtom.sh` | TOMTOM of the pooled set against itself | hours | optional (shipped table) |
| 6 | `06_universal_regulators.py` | cluster the pooled similarity graph (E ≤ 0.5, p ≤ 0.01); a family is core when it spans ≥ 25 biosamples -> **43 motifs** | 15 min | yes |

Inputs: Zenodo `fig2_core_motif_derivation.tar.gz` unpacked under `$ENHANCER_ROOT` (the ten models' promoter PWMs and weights of every biosample, the expression vectors, and every intermediate of steps 1-6, so the slow steps can be skipped). The last part assigns the reader transcription factors of the 43 motifs from JASPAR (Fig. 2c).

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/setup_work_dir.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
os.environ["ENHANCER_ROOT"], os.environ["SEQ_ROOT"] = ENHANCER_ROOT, SEQ_ROOT   # the scripts in code/ read their data root from here
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
# ---- CONFIGURATION ---------------------------------------------------------------------------------------------
BSSID = "BSS00762"              # the biosample used for the per-biosample demonstration (step 3)
RUN_STEP3_FOR_BSSID = False     # True: run step 3 for BSSID. WARNING: 1-3 h (graph clustering) even with the shipped TOMTOM tables; hours more without them
RUN_STEP5_TOMTOM    = False     # True: recompute the self-TOMTOM of the 8,441 pooled motifs. WARNING: several hours; False uses the shipped table
MP = f"{CODE}/motif_pipeline"; PH = f"{CODE}/posthoc_analysis"; RES = f"{MP}/results"
WORK = f"{ENHANCER_ROOT}/notebook_runs/fig2"; os.makedirs(WORK, exist_ok=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
sys.path.insert(0, f"{CODE}/motif_extraction"); from extract_motifs_lib import read_meme
def same_pwms(a, b):
    key = lambda m: np.round(m, 6).tobytes(); ka, kb = set(map(key, a)), set(map(key, b))
    return f"{len(ka & kb)} of {len(kb)} released PWMs recovered" + (" (identical set)" if len(a) == len(b) and ka == kb else "")
def show_pwms(pwms, titles, suptitle=""):
    fig, axes = plt.subplots(1, len(pwms), figsize=(2.0*len(pwms), 1.3))
    for ax, p, t in zip(np.atleast_1d(axes), pwms, titles):
        logomaker.Logo(logomaker.transform_matrix(pd.DataFrame(p, columns=list("ACGT")), from_type="probability", to_type="information"), ax=ax); ax.set_title(t, fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(suptitle, fontsize=9); plt.tight_layout(); plt.show()
bssids = [b for b in open(f"{ENHANCER_ROOT}/Epimap_BSSID_nondup.txt").read().split() if b not in ("BSS01297", "BSS01325")]
print(len(bssids), "biosamples |", "per-model PWM files present:", len(glob.glob(f"{ENHANCER_ROOT}/cell_type_specific_CNN/*/*_pro_motifs_all_256.meme")), "(1,160 expected) |",
      "shipped step-3 sets:", len(glob.glob(f"{RES}/cell_motifs/*/*_cell_motifs.meme")))
shutil.copy(f"{RES}/universal_regulator_motifs.meme", f"{WORK}/universal_regulator_motifs.released.meme")   # keep the released 43 for the comparison at the end

## 4.1 Step 1: the IC-filtered, highest-weight filters of every model
For every biosample and each of its ten models: rank the 2,560 promoter filters by contextual weight, keep the top 750 whose PWM has more than 10 bits of information, and write them with global ids `model*2560 + filter` -> `results/cell_motifs/{BSSID}/{BSSID}_ICfiltered.meme`. The released files are overwritten by identical ones.

In [ ]:
run("python 01_ic_filter.py", cwd=MP)
n = [n_motifs(f"{RES}/cell_motifs/{b}/{b}_ICfiltered.meme") for b in bssids]
print(f"IC-filtered PWMs per biosample: min {min(n)}, median {int(np.median(n))}, max {max(n)} | {BSSID}: {n_motifs(f'{RES}/cell_motifs/{BSSID}/{BSSID}_ICfiltered.meme')}")

## 4.2 Step 3: one biosample's representative motifs
`03_cell_pipeline.py BSSID` (i) runs TOMTOM of the ≤ 7,500 PWMs against themselves (`-dist pearson -min-overlap 7 -evalue -thresh 1`; hours) and against JASPAR 2022, (ii) links pairs with E ≤ 0.01 and p ≤ 0.01 and clusters the graph so that every pair inside a cluster is within path distance 0.01 (all-pairs shortest paths; 1-3 h), (iii) keeps clusters that match a JASPAR motif (E ≤ 0.05, p ≤ 0.01) whose transcription factor is expressed in the biosample (log10 FPKM > 0.5), merges clusters sharing a TF, and (iv) writes one representative PWM per cluster. The script reuses the shipped TOMTOM tables when they exist, so with `RUN_STEP3_FOR_BSSID = True` only the clustering runs (**1-3 h**). By default the shipped result is inspected.

In [ ]:
CM = f"{RES}/cell_motifs/{BSSID}"
if RUN_STEP3_FOR_BSSID:
    shutil.copy(f"{CM}/{BSSID}_cell_motifs.meme", f"{WORK}/{BSSID}_cell_motifs.released.meme")
    run(f"python 03_cell_pipeline.py {BSSID}", cwd=MP)
    print(same_pwms(read_meme(f"{CM}/{BSSID}_cell_motifs.meme"), read_meme(f"{WORK}/{BSSID}_cell_motifs.released.meme")))
tt = pd.read_csv(f"{CM}/{BSSID}_self_tomtom.txt", sep="\t", comment="#"); print(f"self-TOMTOM of {BSSID}: {len(tt):,} pairs, {((tt['E-value'] <= 0.01) & (tt['p-value'] <= 0.01)).sum():,} at E <= 0.01 and p <= 0.01"); print(tt.head(3).to_string())
jt = pd.read_csv(f"{CM}/{BSSID}_jaspar_tomtom.txt", sep="\t", comment="#"); print(f"JASPAR TOMTOM: {jt['Query_ID'].nunique():,} of the PWMs match a JASPAR motif at E <= 0.05, p <= 0.01: {((jt['E-value'] <= 0.05) & (jt['p-value'] <= 0.01)).groupby(jt['Query_ID']).any().sum():,}")
rep = read_meme(f"{CM}/{BSSID}_cell_motifs.meme"); print(f"{BSSID}: {len(rep)} representative motifs"); show_pwms(rep[:6], [f"rep {i}" for i in range(6)], f"first six representative motifs of {BSSID}")
cnt = [n_motifs(f"{RES}/cell_motifs/{b}/{b}_cell_motifs.meme") for b in bssids]; print(f"representatives per biosample over the 116: median {int(np.median(cnt))}, total {sum(cnt):,}")

## 4.3 Steps 4-6: pool the representatives of the 116 biosamples and find the families that recur
Step 4 renames every representative `{BSSID}_feature_{i}` and pools them (8,441 PWMs). Step 5 compares the pooled set with itself by TOMTOM (E ≤ 1 reported; **hours**, shipped). Step 6 links pairs with E ≤ 0.5 and p ≤ 0.01, clusters the graph (path distance ≤ 0.5) and keeps every family that spans at least 25 biosamples: the 43 core motifs, written as `universal_0 … universal_42` (the family rank), with the coverage curve of Fig. 2b.

In [ ]:
run("python 04_collect_features.py", cwd=MP); print("pooled motifs:", n_motifs(f"{RES}/all_feature_motifs.meme"))
if RUN_STEP5_TOMTOM:
    run(f"tomtom -no-ssc -verbosity 1 -text -dist pearson -min-overlap 7 -evalue -thresh 1 {RES}/all_feature_motifs.meme {RES}/all_feature_motifs.meme > {RES}/all_feature_self_tomtom.txt", cwd=MP)
else: print("using the shipped self-TOMTOM table:", f"{RES}/all_feature_self_tomtom.txt", f"({os.path.getsize(f'{RES}/all_feature_self_tomtom.txt')/1e6:.0f} MB)")
run("python 06_universal_regulators.py", cwd=MP)
core = read_meme(f"{RES}/universal_regulator_motifs.meme"); rel = read_meme(f"{WORK}/universal_regulator_motifs.released.meme")
print(f"core motifs regenerated: {len(core)} | {same_pwms(core, rel)}")
from IPython.display import Image, display; display(Image(f"{RES}/plots/universal_group_coverage.png", width=420))
show_pwms(core[:8], [f"universal_{i}" for i in range(8)], "the eight most widely shared core motifs")
shutil.copy(f"{RES}/universal_regulator_motifs.meme", f"{WORK}/universal_regulator_motifs.meme"); print("written:", f"{WORK}/universal_regulator_motifs.meme")

## 4.4 Which transcription factors read them (Fig. 2c)
TOMTOM of the 43 motifs against JASPAR 2022 (`01_jaspar_tomtom.sh`; a minute), then `01_jaspar_analysis.py` assigns every matching JASPAR motif (E ≤ 0.05, p ≤ 0.01) and its transcription factor to the core motif -> `results/step06_JASPAR_assignments.json` (324 motif-to-factor assignments, 287 distinct transcription factors, a median of four per motif) and the pool of JASPAR motifs that resemble no core motif, from which the control sets are drawn (seeded, so the shipped files are reproduced exactly).

In [ ]:
run(f"tomtom -no-ssc -verbosity 1 -text -dist pearson -min-overlap 7 -evalue -thresh 1 {RES}/universal_regulator_motifs.meme {REPO}/data/motifs/JASPAR2022_CORE_vertebrates_non_redundant_unique_names.meme > results/step06_jaspar_tomtom.txt", cwd=PH)
run("python 01_jaspar_analysis.py", cwd=PH)
A = json.load(open(f"{PH}/results/step06_JASPAR_assignments.json")); print(sum(len(v) for v in A.values()), "assignments over", len(A), "motifs;", len({tf for v in A.values() for tf in v}), "distinct TFs (paper: 287)")
for k in list(A)[:8]: print(f"  {k:13s} {', '.join(A[k][:8])}{' ...' if len(A[k]) > 8 else ''}")

## 4.5 The control motif sets
Every later comparison uses **ten random sets of 43 JASPAR motifs that resemble no core motif**. `01_jaspar_analysis.py` above wrote the candidate pool (JASPAR motifs with no TOMTOM similarity to any core motif); `generate_10_controls.py` draws the ten sets from it with fixed seeds -> `results/control_motifs_set_{0..9}.meme` (used by notebooks 06, 07, 08). Seeded, so the shipped sets are reproduced exactly.

In [ ]:
import hashlib
before = {i: hashlib.md5(open(f"{PH}/results/control_motifs_set_{i}.meme", "rb").read()).hexdigest() for i in range(10)}
run("python generate_10_controls.py", cwd=PH)
after = {i: hashlib.md5(open(f"{PH}/results/control_motifs_set_{i}.meme", "rb").read()).hexdigest() for i in range(10)}
print("candidate pool:", sum(1 for l in open(f"{PH}/results/control_candidate_pool.txt") if l.strip() and not l.startswith("#")), "JASPAR motifs |", "ten sets of", n_motifs(f"{PH}/results/control_motifs_set_0.meme"), "| identical to the shipped sets:", before == after)

The 43 PWMs are in `$ENHANCER_ROOT/notebook_runs/fig2/universal_regulator_motifs.meme` (and `code/motif_pipeline/results/`); notebook 05 trains the CNN-CR that uses them as its first-layer filters.